# EDA tổng quan dữ liệu bán hàng

Notebook này tập trung **khám phá và trực quan hóa** các CSV đã có trong `data/processed/`; không chạy lại bước làm sạch và không ghi đè dữ liệu. Chạy lần lượt từ trên xuống dưới sau khi đã chạy pipeline `python scripts/clean_data.py`.

Các bảng nguồn có phạm vi và mục đích khác nhau (đơn hàng Superstore, RFM, Walmart, BigMart...). Không gộp các bảng khác grain thành một bảng chung. Fact bán hàng có cột `Data Source`; nguồn Kaggle 2019–2020 được đánh dấu demo/unverified vì không có provenance giao dịch gốc được xác minh. Kết quả EDA cần được diễn giải từ số liệu thực tế hiển thị sau khi chạy.

## 1. Nạp thư viện và tìm thư mục dự án

Chạy notebook từ thư mục dự án hoặc thư mục con đều được. Nếu `cleaned_data.csv` chưa tồn tại, notebook dừng với thông báo rõ ràng.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in candidates if (path / "data" / "processed" / "cleaned_data.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data/processed/cleaned_data.csv. "
        "Hãy mở notebook từ thư mục dự án và chạy scripts/clean_data.py trước."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
print(f"Thư mục dự án: {PROJECT_ROOT}")
print(f"Thư mục dữ liệu đã xử lý: {PROCESSED_DIR}")

## 2. Đọc các bảng đã xử lý

Bảng `cleaned_data.csv` là fact đơn hàng chính. Các bảng phụ được nạp riêng để mô tả và trực quan hóa theo đúng mục đích của từng bảng.

In [ ]:
csv_paths = sorted(PROCESSED_DIR.glob("*.csv"))
tables = {path.stem: pd.read_csv(path, low_memory=False) for path in csv_paths}
if "cleaned_data" not in tables:
    raise FileNotFoundError(f"Thiếu bảng bắt buộc: {PROCESSED_DIR / 'cleaned_data.csv'}")

orders = tables["cleaned_data"]
print(f"Đã nạp {len(tables)} bảng: {', '.join(tables)}")
print(f"Fact chính: {len(orders):,} dòng, {len(orders.columns)} cột")

## 3. Tổng quan các bảng

Bảng dưới đây được tính tự động từ các CSV hiện có. Ngày nhỏ nhất/lớn nhất chỉ được điền khi tìm thấy cột ngày có thể parse.

In [ ]:
catalog_rows = []
column_rows = []
for table_name, frame in tables.items():
    date_columns = [column for column in frame.columns if "date" in column.lower() or "month" in column.lower()]
    valid_dates = pd.Series(dtype="datetime64[ns]")
    for column in date_columns:
        parsed = pd.to_datetime(frame[column], errors="coerce")
        if parsed.notna().any():
            valid_dates = parsed.dropna()
            break
    catalog_rows.append(
        {
            "Bảng": table_name,
            "Số dòng": len(frame),
            "Số cột": len(frame.columns),
            "Số ô thiếu": int(frame.isna().sum().sum()),
            "Số dòng trùng hoàn toàn": int(frame.duplicated().sum()),
            "Ngày đầu": valid_dates.min() if not valid_dates.empty else pd.NaT,
            "Ngày cuối": valid_dates.max() if not valid_dates.empty else pd.NaT,
            "Dung lượng (MB)": round((PROCESSED_DIR / f"{table_name}.csv").stat().st_size / (1024**2), 2),
        }
    )
    for column in frame.columns:
        column_rows.append(
            {
                "Bảng": table_name,
                "Cột": column,
                "Kiểu dữ liệu": str(frame[column].dtype),
                "Số giá trị thiếu": int(frame[column].isna().sum()),
                "Số giá trị duy nhất": int(frame[column].nunique(dropna=True)),
            }
        )

table_overview = pd.DataFrame(catalog_rows).sort_values("Bảng").reset_index(drop=True)
column_overview = pd.DataFrame(column_rows).sort_values(["Bảng", "Cột"]).reset_index(drop=True)
display(table_overview)
display(column_overview)

## 4. Kiểm tra hợp đồng và chất lượng fact đơn hàng

Các phép kiểm tra chỉ báo cáo hiện trạng, không tự loại dòng. Profit âm được giữ lại vì có thể là đơn lỗ thật. Không xóa trùng theo riêng `Order ID`, vì một đơn có thể có nhiều dòng sản phẩm.

In [ ]:
required_columns = [
    "Order ID", "Order Date", "Customer ID", "Country", "Region",
    "Category", "Sub-Category", "Sales", "Quantity", "Profit",
]
missing_contract_columns = [column for column in required_columns if column not in orders.columns]
quality_rows = [{"Kiểm tra": "Thiếu cột theo hợp đồng", "Số lượng": len(missing_contract_columns), "Chi tiết": ", ".join(missing_contract_columns) or "Không"}]
if not missing_contract_columns:
    parsed_order_dates = pd.to_datetime(orders["Order Date"], errors="coerce")
    quality_rows.extend(
        [
            {"Kiểm tra": "Giá trị thiếu toàn bảng", "Số lượng": int(orders[required_columns].isna().sum().sum()), "Chi tiết": "Các trường thuộc schema chuẩn"},
            {"Kiểm tra": "Order Date không parse được", "Số lượng": int(parsed_order_dates.isna().sum()), "Chi tiết": "Order Date"},
            {"Kiểm tra": "Sales < 0", "Số lượng": int((pd.to_numeric(orders["Sales"], errors="coerce") < 0).sum()), "Chi tiết": "Theo hợp đồng hiện tại Sales >= 0"},
            {"Kiểm tra": "Quantity <= 0", "Số lượng": int((pd.to_numeric(orders["Quantity"], errors="coerce") <= 0).sum()), "Chi tiết": "Theo hợp đồng hiện tại Quantity > 0"},
            {"Kiểm tra": "Profit < 0", "Số lượng": int((pd.to_numeric(orders["Profit"], errors="coerce") < 0).sum()), "Chi tiết": "Không tự động xem là lỗi"},
            {"Kiểm tra": "Dòng trùng hoàn toàn", "Số lượng": int(orders.duplicated().sum()), "Chi tiết": "Không xóa trong notebook"},
        ]
    )
quality_report = pd.DataFrame(quality_rows)
display(quality_report)
if missing_contract_columns:
    raise ValueError(f"Fact chính không đủ schema: {missing_contract_columns}")

## 5. Độ phủ thời gian và các tháng bị khuyết

Biểu đồ chỉ thể hiện tháng thực sự có trong fact. Tháng 2019–2020 hiện được bổ sung bằng nguồn Kaggle demo/unverified; không xem đó là số liệu lịch sử đã xác minh. Tháng nào không có dòng nguồn vẫn là thiếu dữ liệu, không phải doanh thu bằng 0; không nội suy hoặc tự tạo giao dịch.

In [ ]:
orders["Order Date"] = pd.to_datetime(orders["Order Date"], errors="coerce")
valid_orders = orders.dropna(subset=["Order Date"]).copy()
monthly_observed = (
    valid_orders.assign(Month=valid_orders["Order Date"].dt.to_period("M"))
    .groupby("Month", observed=True)
    .agg(Transactions=("Order ID", "nunique"), Rows=("Order ID", "size"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .sort_index()
)
all_months = pd.period_range(monthly_observed.index.min(), monthly_observed.index.max(), freq="M")
missing_months = all_months.difference(monthly_observed.index)
print(f"Khoảng ngày quan sát: {valid_orders['Order Date'].min().date()} đến {valid_orders['Order Date'].max().date()}")
print(f"Số tháng có giao dịch: {len(monthly_observed):,}; số tháng không xuất hiện trong fact: {len(missing_months):,}")
print("Các tháng thiếu:", ", ".join(map(str, missing_months)) if len(missing_months) else "Không có")
monthly_chart = monthly_observed.reindex(all_months)
monthly_chart.index.name = "Month"
monthly_chart = monthly_chart.reset_index()
monthly_chart["Month"] = monthly_chart["Month"].dt.to_timestamp()
fig = px.line(monthly_chart, x="Month", y="Sales", title="Doanh thu theo tháng quan sát được", markers=True)
fig.update_layout(xaxis_title="Tháng", yaxis_title="Sales")
fig.update_traces(connectgaps=False)
fig.show()

## 6. Chỉ số tổng quan fact đơn hàng

Doanh thu, lợi nhuận, số đơn, khách hàng, quốc gia và giá trị đơn được tính trực tiếp từ fact đã nạp.

In [ ]:
kpis = pd.DataFrame(
    [
        {"Chỉ số": "Số dòng sản phẩm", "Giá trị": len(orders)},
        {"Chỉ số": "Số đơn hàng duy nhất", "Giá trị": orders["Order ID"].nunique(dropna=True)},
        {"Chỉ số": "Số khách hàng duy nhất", "Giá trị": orders["Customer ID"].nunique(dropna=True)},
        {"Chỉ số": "Số quốc gia", "Giá trị": orders["Country"].nunique(dropna=True)},
        {"Chỉ số": "Số khu vực", "Giá trị": orders["Region"].nunique(dropna=True)},
        {"Chỉ số": "Tổng doanh thu (Sales)", "Giá trị": orders["Sales"].sum()},
        {"Chỉ số": "Tổng lợi nhuận (Profit)", "Giá trị": orders["Profit"].sum()},
        {"Chỉ số": "Sales trung bình mỗi dòng", "Giá trị": orders["Sales"].mean()},
    ]
)
display(kpis)

if "Data Source" in orders.columns:
    source_summary = orders.groupby("Data Source", as_index=False).agg(
        Rows=("Order ID", "size"), Orders=("Order ID", "nunique"),
        Customers=("Customer ID", "nunique"), Sales=("Sales", "sum"), Profit=("Profit", "sum")
    )
    display(source_summary)
    fig = px.bar(source_summary, x="Data Source", y="Rows", color="Data Source", title="Số dòng theo nguồn dữ liệu")
    fig.update_layout(showlegend=False, xaxis_title="Nguồn dữ liệu")
    fig.show()
else:
    print("Thiếu cột Data Source; chưa thể phân tích provenance.")

## 7. Xu hướng theo năm và tháng

Các năm/tháng không có trong nguồn sẽ không được xem là mức bán bằng 0.

In [ ]:
trend_orders = valid_orders.assign(Year=valid_orders["Order Date"].dt.year, MonthNumber=valid_orders["Order Date"].dt.month)
yearly = trend_orders.groupby("Year", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
display(yearly)
fig = px.bar(yearly, x="Year", y=["Sales", "Profit"], barmode="group", title="Sales và Profit theo năm")
fig.show()

monthly_seasonality = trend_orders.groupby("MonthNumber", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
fig = px.line(monthly_seasonality, x="MonthNumber", y=["Sales", "Profit"], markers=True, title="Tổng Sales và Profit theo tháng trong năm")
fig.update_layout(xaxis_title="Tháng trong năm")
fig.show()

## 8. So sánh địa lý và danh mục sản phẩm

Dùng cùng fact đơn hàng để tổng hợp theo Country, Region, Category và Sub-Category.

In [ ]:
country_summary = orders.groupby("Country", dropna=False, as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
display(country_summary.sort_values("Sales", ascending=False).head(15))
fig = px.bar(country_summary.nlargest(15, "Sales"), x="Sales", y="Country", orientation="h", color="Profit", title="15 quốc gia có Sales cao nhất", hover_data=["Orders"])
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

region_summary = orders.groupby("Region", dropna=False, as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
display(region_summary.sort_values("Sales", ascending=False))
fig = px.bar(region_summary, x="Region", y=["Sales", "Profit"], barmode="group", title="Sales và Profit theo Region")
fig.show()

category_summary = orders.groupby("Category", dropna=False, as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Quantity=("Quantity", "sum"))
display(category_summary.sort_values("Sales", ascending=False))
fig = px.bar(category_summary, x="Category", y=["Sales", "Profit"], barmode="group", title="Sales và Profit theo Category")
fig.show()

subcategory_summary = orders.groupby("Sub-Category", dropna=False, as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
fig = px.bar(subcategory_summary.nlargest(15, "Sales"), x="Sales", y="Sub-Category", orientation="h", color="Profit", title="15 Sub-Category có Sales cao nhất")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 9. Phân phối Sales/Profit và các dòng lợi nhuận âm

Giá trị ngoại lệ theo IQR là dấu hiệu cần xem xét, không mặc định là lỗi dữ liệu.

In [ ]:
for metric in ["Sales", "Profit", "Quantity"]:
    q1 = orders[metric].quantile(0.25)
    q3 = orders[metric].quantile(0.75)
    iqr = q3 - q1
    outlier_count = int(((orders[metric] < q1 - 1.5 * iqr) | (orders[metric] > q3 + 1.5 * iqr)).sum())
    print(f"{metric}: median={orders[metric].median():,.2f}; IQR=[{q1:,.2f}, {q3:,.2f}]; số dòng ngoài ngưỡng IQR={outlier_count:,}")
    fig = px.histogram(orders, x=metric, nbins=50, title=f"Phân phối {metric}")
    fig.show()

loss_orders = orders.loc[orders["Profit"] < 0]
print(f"Số dòng Profit âm: {len(loss_orders):,} ({len(loss_orders) / len(orders):.2%} số dòng)")
loss_by_category = loss_orders.groupby("Category", as_index=False).agg(LossRows=("Profit", "size"), TotalProfit=("Profit", "sum"))
display(loss_by_category.sort_values("TotalProfit"))
fig = px.bar(loss_by_category, x="Category", y="TotalProfit", color="LossRows", title="Profit âm theo Category")
fig.show()
display(loss_orders.sort_values("Profit").head(20))

## 10. Khám phá các bảng bàn giao bổ sung

Mỗi bảng được phân tích độc lập. Nếu một bảng hoặc cột không tồn tại, cell sẽ bỏ qua biểu đồ tương ứng và in rõ lý do.

In [ ]:
if "rfm_customers" in tables:
    rfm = tables["rfm_customers"]
    print(f"RFM: {len(rfm):,} khách hàng, các cột: {list(rfm.columns)}")
    if "Segment" in rfm.columns:
        segment_counts = rfm["Segment"].value_counts(dropna=False).rename_axis("Segment").reset_index(name="Customers")
        display(segment_counts)
        fig = px.bar(segment_counts, x="Segment", y="Customers", title="Số khách hàng theo phân khúc RFM")
        fig.show()
    if {"Frequency", "Monetary"}.issubset(rfm.columns):
        color_column = "Segment" if "Segment" in rfm.columns else None
        fig = px.scatter(rfm, x="Frequency", y="Monetary", color=color_column, hover_data=[column for column in ["Customer ID", "Country"] if column in rfm.columns], title="Frequency và Monetary theo khách hàng")
        fig.show()
else:
    print("Bỏ qua RFM: chưa có rfm_customers.csv trong data/processed.")

if "monthly_sales" in tables:
    monthly_sales = tables["monthly_sales"]
    date_column = next((column for column in monthly_sales.columns if "date" in column.lower() or "month" in column.lower()), None)
    if date_column and "Sales" in monthly_sales.columns:
        monthly_sales[date_column] = pd.to_datetime(monthly_sales[date_column], errors="coerce")
        plot_data = monthly_sales.dropna(subset=[date_column]).sort_values(date_column)
        color_column = "Data Source" if "Data Source" in plot_data.columns else None
        fig = px.line(plot_data, x=date_column, y="Sales", color=color_column, title="Bảng monthly_sales: Sales theo tháng và nguồn", markers=True)
        fig.show()
        print("Lưu ý: các tháng không có nguồn trong bảng tổng hợp không phải dữ liệu quan sát bằng 0.")
    else:
        print(f"Bỏ qua biểu đồ monthly_sales: không tìm thấy cột ngày/tháng và Sales. Cột hiện có: {list(monthly_sales.columns)}")

if "orders_enriched" in tables:
    enriched = tables["orders_enriched"]
    print(f"orders_enriched: {len(enriched):,} dòng; Manager thiếu={enriched['Manager'].isna().sum():,}" if "Manager" in enriched.columns else f"orders_enriched: {len(enriched):,} dòng")
    if {"Region", "Manager"}.issubset(enriched.columns):
        manager_coverage = enriched.groupby("Region", dropna=False, as_index=False).agg(Rows=("Region", "size"), Managers=("Manager", "nunique"))
        display(manager_coverage)

if "walmart_weekly_enriched" in tables:
    walmart = tables["walmart_weekly_enriched"]
    print(f"Walmart đã JOIN: {len(walmart):,} dòng; các cột: {list(walmart.columns)}")
    if {"Store", "Weekly_Sales"}.issubset(walmart.columns):
        store_sales = walmart.groupby("Store", as_index=False).agg(Weekly_Sales=("Weekly_Sales", "sum"))
        fig = px.bar(store_sales, x="Store", y="Weekly_Sales", title="Tổng Weekly_Sales theo cửa hàng Walmart")
        fig.show()

if "bigmart_cleaned" in tables:
    bigmart = tables["bigmart_cleaned"]
    print(f"BigMart: {len(bigmart):,} dòng; các cột: {list(bigmart.columns)}")
    outlet_column = next((column for column in ["Outlet_Type", "Outlet_Identifier"] if column in bigmart.columns), None)
    if outlet_column and "Item_Outlet_Sales" in bigmart.columns:
        outlet_sales = bigmart.groupby(outlet_column, as_index=False).agg(Item_Outlet_Sales=("Item_Outlet_Sales", "sum"))
        fig = px.bar(outlet_sales, x=outlet_column, y="Item_Outlet_Sales", title=f"BigMart: doanh số theo {outlet_column}")
        fig.show()

## 11. Gợi ý ghi nhận kết quả cho báo cáo/Dashboard

Sau khi chạy notebook, ghi lại từ các bảng/biểu đồ thực tế: (1) quy mô và phạm vi thời gian của từng bảng; (2) các tháng/năm nguồn không có dữ liệu; (3) quốc gia, khu vực và danh mục đóng góp Sales/Profit lớn hoặc âm; (4) phân phối, dòng có Profit âm và các ngoại lệ cần giải thích; (5) giới hạn của từng nguồn và của các bảng JOIN.

Không diễn giải khoảng trống thời gian là doanh thu bằng 0, không nội suy để lấp thiếu dữ liệu, và không kết luận nguyên nhân kinh doanh chỉ từ tương quan trong biểu đồ. Nguồn Kaggle được dùng để minh họa khoảng 2019–2020 nhưng provenance giao dịch chưa được xác minh. So sánh kết quả theo `Data Source`, trích dẫn nguồn/license, và không diễn giải doanh số của nguồn này như số liệu thực đã kiểm chứng.